In [ ]:
# tune_models.py
import numpy as np
import optuna
import lightgbm as lgb

from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from data_config import load_data, get_folds, RANDOM_STATE, N_SPLITS

# ================== DATA ==================
X, y, X_test = load_data()
folds = get_folds(y)

# ================== SELECTOR LGBM ==================
def select_top_features_lgb(X_tr, y_tr, n_top=80):
    selector = lgb.LGBMClassifier(
        n_estimators=400,
        num_leaves=64,
        max_depth=-1,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=1,
        random_state=RANDOM_STATE,
    )
    selector.fit(X_tr, y_tr)

    importances = selector.feature_importances_
    idx_sorted = np.argsort(importances)[::-1]
    top_idx = idx_sorted[:n_top]
    top_features = X_tr.columns[top_idx]
    return top_features


# ================== 1) LIGHTGBM ==================
def create_lgb_params(trial):
    return {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": trial.suggest_categorical("boosting_type", ["gbdt", "dart"]),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 150),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 0.9),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 20.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),  # giảm bớt để đỡ chậm
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.5, 2.5),
        "n_jobs": 1,
        "random_state": RANDOM_STATE,
    }


def objective_lgb(trial):
    params = create_lgb_params(trial)
    oof_pred = np.zeros(len(X))
    oof_y = np.array(y)

    for tr_idx, va_idx in folds.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="auc",
        )
        prob_va = model.predict_proba(X_va)[:, 1]
        oof_pred[va_idx] = prob_va

    auc = roc_auc_score(oof_y, oof_pred)
    return auc


# ================== 2) LOGISTIC + SELECTOR ==================
def objective_logreg(trial):
    n_top = trial.suggest_int("n_top_features", 30, min(160, X.shape[1]))
    C = trial.suggest_float("C", 1e-3, 10.0, log=True)

    oof_pred = np.zeros(len(X))

    for tr_idx, va_idx in folds.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        top_features = select_top_features_lgb(X_tr, y_tr, n_top=n_top)
        X_tr_sel = X_tr[top_features]
        X_va_sel = X_va[top_features]

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                C=C,
                penalty="l2",
                solver="lbfgs",
                max_iter=1000,
                class_weight="balanced",
                n_jobs=1
            ))
        ])
        pipe.fit(X_tr_sel, y_tr)
        prob_va = pipe.predict_proba(X_va_sel)[:, 1]
        oof_pred[va_idx] = prob_va

    auc = roc_auc_score(y, oof_pred)
    return auc


# ================== 3) XGBOOST + SELECTOR ==================
def objective_xgb(trial):
    n_top = trial.suggest_int("n_top_features", 30, min(160, X.shape[1]))

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.5, 2.5),
    }

    oof_pred = np.zeros(len(X))

    for tr_idx, va_idx in folds.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        top_features = select_top_features_lgb(X_tr, y_tr, n_top=n_top)
        X_tr_sel = X_tr[top_features]
        X_va_sel = X_va[top_features]

        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="auc",
            nthread=1,
            use_label_encoder=False,
            **params,
        )
        model.fit(
            X_tr_sel, y_tr,
            eval_set=[(X_va_sel, y_va)],
            verbose=False,
        )
        prob_va = model.predict_proba(X_va_sel)[:, 1]
        oof_pred[va_idx] = prob_va

    auc = roc_auc_score(y, oof_pred)
    return auc


# ================== 4) RANDOM FOREST + SELECTOR ==================
def objective_rf(trial):
    n_top = trial.suggest_int("n_top_features", 30, min(160, X.shape[1]))

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
    }

    oof_pred = np.zeros(len(X))

    for tr_idx, va_idx in folds.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        top_features = select_top_features_lgb(X_tr, y_tr, n_top=n_top)
        X_tr_sel = X_tr[top_features]
        X_va_sel = X_va[top_features]

        model = RandomForestClassifier(
            n_jobs=1,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            **params,
        )
        model.fit(X_tr_sel, y_tr)
        prob_va = model.predict_proba(X_va_sel)[:, 1]
        oof_pred[va_idx] = prob_va

    auc = roc_auc_score(y, oof_pred)
    return auc


# ================== MAIN ==================
if __name__ == "__main__":
    # Bạn CHỌN cái nào muốn chạy thì mở comment cái đó thôi để đỡ tốn CPU

    # ---- LightGBM ----
    # study_lgb = optuna.create_study(direction="maximize", study_name="lgb_study")
    # study_lgb.optimize(objective_lgb, n_trials=30)   # giảm n_trials nếu bị giới hạn CPU
    # print("=== LightGBM ===")
    # print("Best AUC:", study_lgb.best_value)
    # print("Best params:", study_lgb.best_params)

    # ---- Logistic Regression ----
    # study_logreg = optuna.create_study(direction="maximize", study_name="logreg_study")
    # study_logreg.optimize(objective_logreg, n_trials=30)
    # print("=== Logistic Regression ===")
    # print("Best AUC:", study_logreg.best_value)
    # print("Best params:", study_logreg.best_params)

    # ---- XGBoost ----
    # study_xgb = optuna.create_study(direction="maximize", study_name="xgb_study")
    # study_xgb.optimize(objective_xgb, n_trials=30)
    # print("=== XGBoost ===")
    # print("Best AUC:", study_xgb.best_value)
    # print("Best params:", study_xgb.best_params)

    # ---- RandomForest ----
    # study_rf = optuna.create_study(direction="maximize", study_name="rf_study")
    # study_rf.optimize(objective_rf, n_trials=30)
    # print("=== RandomForest ===")
    # print("Best AUC:", study_rf.best_value)
    # print("Best params:", study_rf.best_params)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Giả sử X_sel là DataFrame sau khi bạn đã chọn feature (VD: top 80 từ LGBM)
# X_sel.columns -> chứa cả feature numeric + categorical

# CÁC CỘT MUỐN ONE-HOT
cat_cols = ["cat1", "cat2"]  # thay bằng tên cột thật của bạn

# CÁC CỘT CÒN LẠI (numeric hoặc không one-hot)
num_cols = [c for c in X_sel.columns if c not in cat_cols]

# Bộ tiền xử lý:
# - với cat_cols: OneHotEncoder
# - với num_cols: truyền nguyên (passthrough)
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

# Vì sau OneHotEncoder sẽ ra sparse matrix,
# nên StandardScaler nếu dùng thì nên with_mean=False để không lỗi.
logistic_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("scaler", StandardScaler(with_mean=False)),  # có thể bỏ nếu bạn không cần scale
    ("clf", LogisticRegression(
        C=1.0,
        penalty="l2",
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced",
        n_jobs=1
    ))
])

# Train
logistic_pipeline.fit(X_sel, y)

# Predict proba
y_pred_proba = logistic_pipeline.predict_proba(X_sel)[:, 1]
